<a href="https://colab.research.google.com/github/oumeima-belala/hotel/blob/main/ML_Worshop_QSAR.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 1) Data Collection

## 1.1) Search ChEMBL Database

### 1.1.1) Install ChEMBL Library

In [ ]:
!pip install chembl_webresource_client

### 1.1.2) Import Libraries

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from chembl_webresource_client.new_client import new_client

### 1.1.3) Search for Target Protein

In [ ]:
target = new_client.target
target_query = target.search('aromatase')
targets = pd.DataFrame.from_dict(target_query)
targets

In [ ]:
selected_target = targets['target_chembl_id'][0]
selected_target

### 1.1.4) Get Target Bioactivities

In [ ]:
activity = new_client.activity
Results = activity.filter(target_chembl_id = selected_target).filter(standard_type = 'IC50')

In [ ]:
data = pd.DataFrame.from_dict(Results)
data

# 2) Data Pre-processing

## 2.1) Handling missing data

### 2.1.1) Delete rows without standard value and SMILES

In [ ]:
(data['standard_value'].notna()).value_counts()

In [ ]:
(data['canonical_smiles'].notna()).value_counts()

In [ ]:
data2 = data[data['standard_value'].notna()]
data2 = data2[data2['canonical_smiles'].notna()]
data2

In [ ]:
data2.reset_index(drop = True, inplace = True)
data2

## 2.2) Delete duplicated SMILES

In [ ]:
len(data2.canonical_smiles.unique())

In [ ]:
data3 = data2.drop_duplicates('canonical_smiles')
data3.reset_index(drop = True, inplace = True)
data3

## 2.3) Select 3 columns : ID, SMILES and IC50

In [ ]:
data3.columns

In [ ]:
selected_columns = ['molecule_chembl_id', 'canonical_smiles', 'standard_value']
data4 = data3[selected_columns]
data4

In [ ]:
data4.rename(columns = {'standard_value':'IC50', 'molecule_chembl_id':'ID', 'canonical_smiles':'SMILES'}, inplace = True)
data4

In [ ]:
data4.info()

In [ ]:
data4 = data4.convert_dtypes()
data4.info()

In [ ]:
data4['IC50'] = data4['IC50'].astype(float)
data4.info()

In [ ]:
(data4['IC50'] == 0).value_counts()

In [ ]:
data4['IC50'].describe()

In [ ]:
data5 = data4[data4['IC50'] != 0]
data5.reset_index(drop = True, inplace = True)
data5

## 2.4) Calculate pIC50

In [ ]:
def pIC50(DF):
  p_values = []
  for i in DF['IC50']:
    if i > 100000000:
      i = 100000000
    molar = i * (10 ** -9)
    p = -np.log10(molar)
    p_values.append(p)
  DF['pIC50'] = p_values
  DF2 = DF.drop('IC50', axis = 1)
  return DF2

In [ ]:
data6 = pIC50(data5)
data6

In [ ]:
data6['pIC50'].describe()

# 3) Exploratory Data Analysis

## 3.1) Install RDKIT Library

In [ ]:
!pip install rdkit

## 3.2) Import modules from RDKIT library

In [ ]:
from rdkit import Chem
from rdkit.Chem import Descriptors, Lipinski

## 3.3) Calculate Lipinski descriptors

In [ ]:
def lipinski(smiles):

    moldata= []
    for elem in smiles:
        mol=Chem.MolFromSmiles(elem)
        moldata.append(mol)

    baseData= np.arange(1,1)
    i=0
    for mol in moldata:

        desc_MolWt = Descriptors.MolWt(mol)
        desc_MolLogP = Descriptors.MolLogP(mol)
        desc_NumHDonors = Lipinski.NumHDonors(mol)
        desc_NumHAcceptors = Lipinski.NumHAcceptors(mol)

        row = np.array([desc_MolWt,
                        desc_MolLogP,
                        desc_NumHDonors,
                        desc_NumHAcceptors])

        if(i==0):
            baseData=row
        else:
            baseData=np.vstack([baseData, row])
        i=i+1

    columnNames=["MW","LogP","NumHDonors","NumHAcceptors"]
    descriptors = pd.DataFrame(data=baseData,columns=columnNames)

    return descriptors

In [ ]:
Smiles = data6['SMILES']
Lipinski_Descriptors = lipinski(Smiles)
Lipinski_Descriptors

In [ ]:
data_combined = pd.concat([data6, Lipinski_Descriptors], axis=1)
data_combined

In [ ]:
data_combined.pIC50.describe()

In [ ]:
Activity = []
for p in data_combined['pIC50']:
  if p >= 6:
    Activity.append('Active')
  else:
    Activity.append('Inactive')
data_combined['Class'] = Activity
data_combined

## 3.4) Generate plots

### 3.4.1) Histograms

In [ ]:
p_values = np.array(data_combined['pIC50'])
n, bin, patches = plt.hist(p_values, 50, density = True,
                           facecolor = 'g', alpha = 0.75)
plt.xlabel('pIC50 values')
plt.ylabel('Molecules number')
plt.title('Histogram of pIC50')
plt.grid(True)
plt.show()

In [ ]:
sns.countplot(data = data_combined, x = 'Class')
plt.xlabel('Class')
plt.ylabel('Molecules number')
plt.title('Histrogram of Activity Class')
plt.show()

In [ ]:
data_combined['Class'].value_counts()

### 3.4.2) Boxplots

In [ ]:
sns.boxplot(x = data_combined['Class'], y = data_combined['MW'])
plt.title("Boxplot on Molecular Weight")
plt.show()

In [ ]:
sns.boxplot(x = data_combined['Class'], y = data_combined['LogP'])
plt.title("Boxplot on LogP")
plt.show()

In [ ]:
sns.boxplot(x = data_combined['Class'], y = data_combined['NumHDonors'])
plt.title("Boxplot on NumHDonors")
plt.show()

In [ ]:
sns.boxplot(x = data_combined['Class'], y = data_combined['NumHAcceptors'])
plt.title("Boxplot on NumHAcceptors")
plt.show()

### 3.4.3) Scatterplot

In [ ]:
g = sns.scatterplot(data = data_combined, x = 'MW', y = 'LogP',
                    hue = 'Class', size = 'pIC50')
sns.move_legend(g, 'upper left',
                bbox_to_anchor = (1, 1), title = 'Class')
plt.title("Scatterplot on Molecular Weight and LogP")
plt.show()

## 3.5) Statistical Test (nonparametric test : Mann-Whitney U Test)

In [ ]:
def mannwhitney(descriptor):
  from numpy.random import seed
  from numpy.random import randn
  from scipy.stats import mannwhitneyu

# seed the random number generator
  seed(1)

# actives and inactives
  selection = [descriptor, 'Class']
  df14 = data_combined[selection]

  active = df14[df14['Class'] == 'Active']
  active = active[descriptor]

  inactive = df14[df14['Class'] == 'Inactive']
  inactive = inactive[descriptor]

# compare samples
  stat, p = mannwhitneyu(active, inactive)
  #print('Statistics=%.3f, p=%.3f' % (stat, p))

# interpret
  alpha = 0.05
  if p > alpha:
    interpretation = 'Same distribution (fail to reject H0)'
  else:
    interpretation = 'Different distribution (reject H0)'

  results = pd.DataFrame({'Descriptor':descriptor,
                          'Statistics':stat,
                          'p':p,
                          'alpha':alpha,
                          'Interpretation':interpretation}, index=[0])
  filename = 'mannwhitneyu_' + descriptor + '.csv'
  results.to_csv(filename)

  return results

In [ ]:
mannwhitney('pIC50')

In [ ]:
mannwhitney('MW')

In [ ]:
mannwhitney('LogP')

In [ ]:
mannwhitney('NumHDonors')

In [ ]:
mannwhitney('NumHAcceptors')

# 4) Descriptors Generation

## 4.1) Install PADELPy Library

In [ ]:
!pip install padelpy

In [ ]:
# Download fingerprint XML files
! wget https://github.com/dataprofessor/padel/raw/main/fingerprints_xml.zip
! unzip fingerprints_xml.zip

In [ ]:
# List and sort fingerprint XML files
import glob
xml_files = glob.glob("*.xml")
xml_files.sort()
xml_files

FP_list = ['AtomPairs2DCount',
 'AtomPairs2D',
 'EState',
 'CDKextended',
 'CDK',
 'CDKgraphonly',
 'KlekotaRothCount',
 'KlekotaRoth',
 'MACCS',
 'PubChem',
 'SubstructureCount',
 'Substructure']

# Create a dictionary
fp = dict(zip(FP_list, xml_files))

df = pd.concat([data_combined['SMILES'], data_combined['ID']], axis=1 )
df.to_csv('molecules.smi', sep='\t', index=False, header=False)
df

## 4.2) Generate Fingerprints

In [ ]:
# To fix the Java JRE 6+ not found error, run this command line
!apt-get install -y openjdk-11-jre-headless

In [ ]:
from padelpy import padeldescriptor
padeldescriptor(mol_dir = "molecules.smi",
             d_file = 'PubChem_descriptors.csv',
             descriptortypes = fp['PubChem'],
             detectaromaticity = True,
             standardizetautomers = True,
             threads = 2,
             removesalt = True,
             log = True,
             fingerprints = True)

In [ ]:
Descriptors = pd.read_csv('PubChem_descriptors.csv')
Descriptors

In [ ]:
Descriptors.drop("Name", axis = 1 , inplace = True)
Descriptors

# 5) Feature Selection

In [ ]:
from sklearn.feature_selection  import VarianceThreshold
selection = VarianceThreshold(threshold = (0.8 * (1 - 0.8)))
X = selection.fit_transform(Descriptors)
X

In [ ]:
Y = data_combined['pIC50']
Y

# 6) Model Training

## 6.1) Split the dataset into Train/Test sets

In [ ]:
from sklearn.model_selection import train_test_split
X_Train, X_Test, Y_Train, Y_Test = train_test_split(X, Y, train_size = 0.8,
                                                    test_size = 0.2)

In [ ]:
X.shape, Y.shape

In [ ]:
X_Train.shape, Y_Train.shape

In [ ]:
X_Test.shape, Y_Test.shape

## 6.2) Train the model

In [ ]:
from sklearn.ensemble import RandomForestRegressor
model = RandomForestRegressor(n_estimators = 10)
model.fit(X_Train, Y_Train)

In [ ]:
Y_Pred = model.predict(X_Test)
Y_Pred

# 7) Model Evaluation

In [ ]:
from sklearn import metrics

In [ ]:
R2 = metrics.r2_score(Y_Test, Y_Pred)
MSE = metrics.mean_squared_error(Y_Test, Y_Pred)
RMSE = np.sqrt(MSE)
MAE = metrics.mean_absolute_error(Y_Test, Y_Pred)
MAPE = metrics.mean_absolute_percentage_error(Y_Test, Y_Pred)
Max_Error = metrics.max_error(Y_Test, Y_Pred)
print("R2 = ", round(R2, 3))
print("MSE = ", round(MSE, 3))
print("RMSE = ", round(RMSE, 3))
print("MAE = ", round(MAE, 3))
print("MAPE = ", round(MAPE, 3))
print("Max_Error = ", round(Max_Error, 3))

In [ ]:
sns.set(color_codes=True)
sns.set_style("white")

ax = sns.regplot(x= Y_Test, y = Y_Pred, scatter_kws={'alpha':0.4})
ax.set_xlabel('Experimental pIC50', fontsize='large', fontweight='bold')
ax.set_ylabel('Predicted pIC50', fontsize='large', fontweight='bold')
ax.set_xlim(0, 12)
ax.set_ylim(0, 12)
ax.figure.set_size_inches(5, 5)
plt.show()

# 8) Feature Importance

In [ ]:
mask = selection.get_support()
selected_columns_indices = [i for i, f in enumerate(mask) if f]
features = Descriptors.columns[selected_columns_indices]
features

In [ ]:
features.shape , len(selected_columns_indices)

In [ ]:
importances = model.feature_importances_
best_indices = np.argsort(importances)[-5:]
worst_indices = np.argsort(importances)[:5]
indices = np.concatenate([worst_indices, best_indices])

plt.figure(figsize=(10,8))
plt.title('10 Most and 10 Least Important Features in the Model')
plt.barh(range(len(indices)), importances[indices], color='b', align='center')
plt.yticks(range(len(indices)), [features[i] for i in indices])
plt.xlabel('Relative Importance')
plt.ylabel('Feature Name')
plt.show()